Import Python modules

In [1]:
import pandas as pd
import numpy as np
import glob

Read in counts data

In [2]:
# Read in data
counts_df = pd.read_csv('../results/counts.csv', keep_default_na=False)
counts_df = counts_df.replace('', np.nan)

# Convert codon_site to string to avoid issues with rows with semicolon-separated codon sites
counts_df['codon_site'] = counts_df['codon_site'].astype(str)

counts_df.head()

/tmp/ipykernel_466608/1834990010.py:2: DtypeWarning: Columns (0: codon_position, 1: codon_site) have mixed types. Specify dtype option on import or set low_memory=False.
  counts_df = pd.read_csv('../results/counts.csv', keep_default_na=False)


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,mut_class,mut_type,subtype,segment,segment_subtype,segment_length,host,evo_opp,rate,exclude_from_mut_rate_analysis
0,1159,A1159C,A,C,PA,1,387,ATT,CTT,I,...,nonsynonymous,AC,all,PA,PA_all,2148,avian,0.496741,0.0,False
1,1162,A1162C,A,C,PA,1,388,AGC,CGC,S,...,nonsynonymous,AC,all,PA,PA_all,2148,all,10.583333,0.09448818897637795,False
2,1162,A1162C,A,C,PA,1,388,AGC,CGC,S,...,nonsynonymous,AC,all,PA,PA_all,2148,all,0.065642,0.0,False
3,1162,A1162C,A,C,PA,1,388,AGC,CGC,S,...,nonsynonymous,AC,all,PA,PA_all,2148,all,2.068901,0.0,False
4,1162,A1162C,A,C,PA,1,388,AGC,CGC,S,...,nonsynonymous,AC,all,PA,PA_all,2148,all,30.415270,0.0657564440090614,False


Read in predicted rates from neutral model

In [3]:
predicted_rates_df = pd.read_csv(
    '../results/expected_rates.csv',
    keep_default_na=False
)
del predicted_rates_df['tau_squared']
predicted_rates_df.head()

,mut_type,segment,motif,predicted_rate
0,AC,HA,AAA,0.200380
1,AC,HA,AAC,0.315456
2,AC,HA,AAG,0.246664
3,AC,HA,AAT,0.308056
4,AC,HA,CAA,0.178563


Compute expected counts and subset the data to mutations with at least X actual or expected counts

In [4]:
counts_cutoff = 10
# Merge counts with predicted rates; do NOT pre-filter on counts here. The
# cutoff is applied separately to the nt-level and AA-level aggregations
# below, so a mutation passes if its summed counts at the level being
# analyzed reach the cutoff. Pre-filtering at the row level would drop
# low-count codon-context rows that, when summed for a synonymous AA
# mutation with multiple nt paths, contribute meaningfully to the AA total
# (and would cause aa_fitness_effects.csv to disagree with the per-subset
# pipeline, which sums all nt rows before filtering).
actual_expected_df = (
    counts_df
    .merge(predicted_rates_df, on=['mut_type', 'segment', 'motif'], how='left')
    .assign(expected_count=lambda x: x['predicted_rate'] * x['evo_opp'])
)
actual_expected_df.to_csv('../results/actual_expected.csv', index=False)
print("Number of merged nt-level rows (no count cutoff applied):", len(actual_expected_df))
actual_expected_df.head()

Number of merged nt-level rows (no count cutoff applied): 2090295


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,subtype,segment,segment_subtype,segment_length,host,evo_opp,rate,exclude_from_mut_rate_analysis,predicted_rate,expected_count
0,1159,A1159C,A,C,PA,1,387,ATT,CTT,I,...,all,PA,PA_all,2148,avian,0.496741,0.0,False,0.277810,0.137999
1,1162,A1162C,A,C,PA,1,388,AGC,CGC,S,...,all,PA,PA_all,2148,all,10.583333,0.09448818897637795,False,0.164151,1.737268
2,1162,A1162C,A,C,PA,1,388,AGC,CGC,S,...,all,PA,PA_all,2148,all,0.065642,0.0,False,0.109968,0.007219
3,1162,A1162C,A,C,PA,1,388,AGC,CGC,S,...,all,PA,PA_all,2148,all,2.068901,0.0,False,0.222445,0.460217
4,1162,A1162C,A,C,PA,1,388,AGC,CGC,S,...,all,PA,PA_all,2148,all,30.415270,0.0657564440090614,False,0.249623,7.592366


Compute fitness effects of single nucleotide mutations.

In [5]:
pseudo_count = 0.5
groupby_cols = [
    'host', 'subtype', 'segment', 'gene', 'site', 'wt_nt', 'mut_nt', 'nt_mut',
    'mut_class'
]
nt_fitness_df = (
    actual_expected_df
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .query("actual_count >= @counts_cutoff | expected_count >= @counts_cutoff")
    .assign(
        delta_fitness=lambda x: \
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)
nt_fitness_df.to_csv('../results/nt_fitness_effects.csv', index=False)
print("Number of unique nt muts with estimated fitness effects:", len(nt_fitness_df))
nt_fitness_df.head()

Number of unique nt muts with estimated fitness effects: 126506


,host,subtype,segment,gene,site,wt_nt,mut_nt,nt_mut,mut_class,actual_count,expected_count,delta_fitness
0,all,H1,HA,HA,2,T,A,T2A,nonsynonymous,0,21.441897,-3.781545
1,all,H1,HA,HA,2,T,C,T2C,nonsynonymous,0,64.250962,-4.863696
2,all,H1,HA,HA,2,T,G,T2G,nonsynonymous,0,13.628131,-3.341315
3,all,H1,HA,HA,3,G,A,G3A,nonsynonymous,0,149.012999,-5.700531
5,all,H1,HA,HA,3,G,T,G3T,nonsynonymous,0,29.345482,-4.089181


Compute average fitness effects of synonymous nucleotide mutations at a given site.

In [6]:
groupby_cols = ['host', 'subtype', 'segment', 'gene', 'site']
site_syn_fitness_df = (
    nt_fitness_df
    .query("mut_class == 'synonymous'")
    .groupby(groupby_cols, as_index=False)
    .agg({'delta_fitness': 'mean'})
)
site_syn_fitness_df.to_csv('../results/sitewise_synonymous_fitness_effects.csv', index=False)
print("Number of sites with estimated synonymous fitness effects:", len(site_syn_fitness_df))
site_syn_fitness_df.head()

Number of sites with estimated synonymous fitness effects: 20985


,host,subtype,segment,gene,site,delta_fitness
0,all,H1,HA,HA,6,0.234975
1,all,H1,HA,HA,9,-0.037904
2,all,H1,HA,HA,12,-0.392103
3,all,H1,HA,HA,13,-0.474938
4,all,H1,HA,HA,15,-0.913559


Compute fitness effects of amino-acid mutations, aggregating data across nucleotide mutations that result in the same amino-acid mutation.

In [7]:
groupby_cols = [
    'host', 'subtype', 'segment', 'gene', 'codon_site', 'wt_aa', 'mut_aa', 'aa_mut',
    'mut_class'
]
aa_fitness_df = (
    actual_expected_df
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .query("actual_count >= @counts_cutoff | expected_count >= @counts_cutoff")
    .assign(
        delta_fitness=lambda x: \
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)
aa_fitness_df.to_csv('../results/aa_fitness_effects.csv', index=False)
print("Number of unique aa muts with estimated fitness effects:", len(aa_fitness_df))
aa_fitness_df.head()

Number of unique aa muts with estimated fitness effects: 110580


,host,subtype,segment,gene,codon_site,wt_aa,mut_aa,aa_mut,mut_class,actual_count,expected_count,delta_fitness
0,all,H1,HA,HA,1,M,I,M1I,nonsynonymous,0,179.518926,-5.886209
1,all,H1,HA,HA,1,M,K,M1K,nonsynonymous,0,21.441897,-3.781545
2,all,H1,HA,HA,1,M,R,M1R,nonsynonymous,0,13.628131,-3.341315
3,all,H1,HA,HA,1,M,T,M1T,nonsynonymous,0,64.250962,-4.863696
5,all,H1,HA,HA,10,C,C,C10C,synonymous,16,12.576026,0.232580


Report the number of amino-acid level mutations with data in each category (including "synonymous" mutations that don't change the amino acid)

In [8]:
aa_fitness_df.groupby(['host', 'mut_class']).size()

host   mut_class    
all    nonsense          2213
       nonsynonymous    33443
       synonymous        7649
avian  nonsense           989
       nonsynonymous    15891
       synonymous        5519
human  nonsense          1796
       nonsynonymous    25212
       synonymous        5490
swine  nonsense           280
       nonsynonymous     8107
       synonymous        3991
dtype: int64

In [9]:
aa_fitness_df[
    (aa_fitness_df['host'] == 'all') &
    (aa_fitness_df['mut_class'] == 'nonsynonymous')
].groupby(['segment', 'subtype']).size()

segment  subtype
HA       H1         2845
         H3         2631
         H5         1519
         H7           32
         H9          917
MP       all        2093
NA       N1         2463
         N2         2481
         N6          363
         N8          365
         N9           17
NP       all        3118
NS       all        2041
PA       all        4086
PB1      all        4034
PB2      all        4438
dtype: int64